# Step 13 — Local JSON Storage Schema
**Tough Talks · Phase 5**

Goal: stand up the on-device storage layer that persists every long-lived artifact the Phase 1–4 components produce, then drive every read / write path end-to-end against a clean filesystem root. Storage is **local-only** — the layer never opens a network socket, and the on-disk layout is the contract the Phase 6 frontend reads / writes via the new `/storage/*` routes.

**Architecture decisions**:

- **Root**: `<repo>/data/local/` by default (gitignored — user data never gets committed). Override via the `TOUGH_TALKS_STORAGE_ROOT` env var for on-device / packaged deployments where the data should live in the user's home directory. The notebook uses a fresh `tempfile.mkdtemp(...)` so the demo never touches a real user's data.
- **Bundled conversation shape**: one JSON per round under `conversations/<round_id>.json` carrying the transcript plus every per-round artifact (premortem / debrief / aftermath / optional emotion_results). The frontend's round-detail view renders in one read. Wrapper schema at `data/schemas/conversation.schema.json` (new in this step) — references the existing per-component schemas as nested objects.
- **API routes added now**: `/storage/*` (GET/PUT/LIST for talk-dna, vault, pulse; full CRUD on conversations). Step 14 (frontend integration) becomes a pure JS wiring task — every backend surface the frontend needs is already there.
- **Schema validation on write, log warnings on read** — every `save_*` runs a lightweight required-fields check against the matching schema before the bytes hit disk; reads log a warning on mismatch but still return the dict (a schema evolution must not lock users out of older payloads). The per-component runtime coercers do the deep validation upstream; storage is the last "did you actually hand me the right shape?" check.
- **Atomic writes**: every save writes through a per-directory `.tmp` file, fsyncs (best-effort), and renames into place. A crash mid-write leaves the previous version intact.

**Layout**

```
data/local/
├── talk_dna/<user_id>.json
├── person_vault/<person_id>.json
├── pulse/<person_id>.json
└── conversations/<round_id>.json
```

**What `done` looks like for this step**

1. `backend/core/_runtime/storage.py` round-trips every entity type via the in-process Python API.
2. Every `/storage/*` route returns 200 on the happy path, 404 on missing records, and 422 on validation failures.
3. Atomic-write invariant holds: no `.tmp` files left behind after success; previous version intact after a simulated failure.
4. The final cell renders a single pass/fail table over every step (Python API + HTTP API). The cell runs unconditionally so a failure on one check doesn't hide the others (`[[rule]] Final cell is always a results table that renders unconditionally`).

Notebook doesn't load a model — every check in this step is filesystem-only. Audio / inference routes stay 503 under the test fixture, which is the right semantics.

In [1]:
# ── 0. Install / upgrade dependencies ────────────────────
# Storage is filesystem-only; the only deps we need on top of what's
# pre-installed in Colab are FastAPI + httpx (TestClient) + pydantic.
# All already pinned in requirements.txt, but Colab doesn't ship them
# pre-installed.

!pip install -q fastapi 'pydantic>=2.6' httpx

In [2]:
# ── 1. Locate (or fetch) the repo, put it on sys.path ─────────
# Same shim as Steps 01–12. The shim also flushes cached backend.*
# modules from sys.modules so running this cell after a git refresh
# is enough to pick up new exports — no kernel restart required.

import os, pathlib, subprocess, sys

REPO_URL  = "https://github.com/EhsanFarazmand/tough_talks.git"
REPO_NAME = "tough_talks"

def _looks_like_repo(p: pathlib.Path) -> bool:
    return (p / "backend" / "core" / "_runtime").is_dir()

def _scan_for_repo() -> pathlib.Path | None:
    cwd = pathlib.Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if _looks_like_repo(parent):
            return parent
    for base in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")):
        candidate = base / REPO_NAME
        if _looks_like_repo(candidate):
            return candidate
    return None

def _refresh(target: pathlib.Path) -> None:
    if not (target / ".git").is_dir():
        return
    print(f"Refreshing {target} from origin")
    subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1", "origin"],
                   capture_output=True, check=False)
    subprocess.run(["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"],
                   capture_output=True, check=False)

REPO_ROOT = _scan_for_repo()
if REPO_ROOT is None:
    base = next((b for b in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")) if b.is_dir()),
                pathlib.Path.cwd())
    target = base / REPO_NAME
    print(f"Cloning {REPO_URL} -> {target}")
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed:\n" + result.stderr)
    REPO_ROOT = target
else:
    _refresh(REPO_ROOT)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

_stale = [m for m in list(sys.modules) if m == "backend" or m.startswith("backend.")]
for _m in _stale:
    del sys.modules[_m]
if _stale:
    print(f"Cleared {len(_stale)} cached backend.* module(s) from sys.modules")

print(f"Repo root: {REPO_ROOT}")

Cloning https://github.com/EhsanFarazmand/tough_talks.git -> /content/tough_talks
Repo root: /content/tough_talks


In [3]:
# ── 2. Imports ──────────────────────────────
import json, tempfile
from pathlib import Path

from fastapi.testclient import TestClient

from backend.api.deps import get_storage_root
from backend.api.main import app
from backend.core._runtime import (
    DEFAULT_STORAGE_ROOT,
    ENV_STORAGE_ROOT,
    StorageError,
    StorageNotFoundError,
    delete_conversation,
    list_conversations,
    list_person_vaults,
    list_pulses,
    load_conversation,
    load_person_vault,
    load_pulse,
    load_talk_dna,
    resolve_storage_root,
    save_conversation,
    save_person_vault,
    save_pulse,
    save_talk_dna,
)

In [4]:
# ── 3. Config ───────────────────────────────
# Use a brand-new tmp dir for every notebook run so the demo never
# stomps on real user data. resolve_storage_root would normally point
# at data/local/ — we surface that path here for visibility but the
# rest of the notebook stays inside STORAGE_ROOT.

STORAGE_ROOT = Path(tempfile.mkdtemp(prefix="step13_storage_"))
print(f"Notebook storage root        : {STORAGE_ROOT}")
print(f"Default production root      : {DEFAULT_STORAGE_ROOT}")
print(f"Env override knob            : {ENV_STORAGE_ROOT}")
print(f"resolve_storage_root() (now) : {resolve_storage_root()}")

Notebook storage root        : /tmp/step13_storage_yglqmt03
Default production root      : /content/tough_talks/data/local
Env override knob            : TOUGH_TALKS_STORAGE_ROOT
resolve_storage_root() (now) : /content/tough_talks/data/local


In [5]:
# ── 4. Fixtures (Jamie + a representative practice round) ──────
# Same Jamie v2 PersonVault shape used across Steps 6–12, plus a
# minimal TalkDNA / Pulse / Conversation. Schema-conformant but kept
# small — the storage layer only does structural validation, not
# semantic. The full Phase 4 fixtures used in Step 12 round-trip
# cleanly through these helpers too; we don't repeat them here
# because the storage layer is shape-agnostic.
#
# user_id is "local" everywhere — the schema description spells out
# the convention ("Always 'local' for on-device — no account needed")
# and the runtime exports it as DEFAULT_USER_ID.

TALK_DNA_PAYLOAD = {
    "user_id": "local",
    "version": 1,
    "conversation_count": 1,
    "patterns": {
        "filler_phrases": ["I just feel like", "kind of"],
        "apology_rate": 0.15,
        "silence_under_pressure": False,
        "sarcasm_frequency": "low",
        "escalation_triggers": ["missed deadlines"],
        "avg_turn_length_words": 18.0,
    },
    "strengths": ["clear_problem_statement", "listens_actively"],
    "weaknesses": ["hedges_before_vulnerable_statements"],
    "updated_at": "2026-05-15T10:00:00+00:00",
}

VAULT_PAYLOAD = {
    "person_id": "person_cf528d57b789",
    "name": "Jamie",
    "relationship_type": "colleague",
    "version": 2,
    "conversation_count": 2,
    "profile": {
        "communication_style": "defensive",
        "emotional_triggers": ["citing past commitments"],
        "de_escalation_keys": ["explicitly disowning blame", "reframing as joint problem-solving"],
        "common_deflections": ["I told you the staging tables weren't done"],
        "responds_best_to": "concrete next steps and shared ownership.",
    },
    "updated_at": "2026-05-14T15:00:00+00:00",
}

PULSE_PAYLOAD = {
    "person_id": "person_cf528d57b789",
    "person_name": "Jamie",
    "relationship_type": "colleague",
    "version": 1,
    "round_count": 2,
    "health_score": 0.65,
    "health_trend": "improving",
    "trajectory_summary": (
        "You moved Jamie from a hard deflection in round A to a concession in round B "
        "by reframing the missed deadline as a joint problem."
    ),
    "round_summaries": [
        {"round_id": "round_a", "started_at": "2026-04-30T16:00:00+00:00",
         "goal_status": "partial", "prediction_accuracy": 0.6,
         "headline": "You drew the deflection out into a shared plan."},
        {"round_id": "round_b", "started_at": "2026-05-14T15:30:00+00:00",
         "goal_status": "achieved", "prediction_accuracy": 0.75,
         "headline": "You committed Jamie to Wednesday EOD with a daily check-in."},
    ],
    "recurring_patterns": [
        {"pattern": "Jamie opens defensively; concedes once you reframe as joint problem-solving.",
         "evidence_round_ids": ["round_a", "round_b"]},
    ],
    "emerging_concerns": [],
    "relationship_wins": [
        {"win": "Joint-problem-solving reframe lands consistently.", "round_id": "round_b"},
    ],
    "next_step_recommendation": "Try the reframe in turn 1 next time instead of turn 3.",
    "updated_at": "2026-05-15T10:30:00+00:00",
}

CONVERSATION_PAYLOAD = {
    "round_id": "round_step13_demo",
    "person_id": "person_cf528d57b789",
    "person_name": "Jamie",
    "started_at": "2026-05-15T15:00:00+00:00",
    "mode": "practice",
    "user_goal": "Get Jamie to commit to Wednesday EOD.",
    "transcript": [
        {"speaker": "user", "text": "I noticed Tuesday's deadline slipped — what's going on?"},
        {"speaker": "persona", "persona_name": "Jamie",
         "reply": "I told you the staging tables weren't done last Friday.",
         "resistance_type": "deflect", "escalation_level": 0.55},
        {"speaker": "user", "text": "I hear that. I'm not blaming you — what's the blocker we can name today?"},
        {"speaker": "persona", "persona_name": "Jamie",
         "reply": "If we agree on a daily Slack check-in I can commit to Wednesday.",
         "resistance_type": "concede", "escalation_level": 0.25},
    ],
    "updated_at": "2026-05-15T15:00:00+00:00",
}

print("Fixtures loaded:",
      f"talk_dna v{TALK_DNA_PAYLOAD['version']},",
      f"vault v{VAULT_PAYLOAD['version']} for {VAULT_PAYLOAD['name']},",
      f"pulse v{PULSE_PAYLOAD['version']} ({PULSE_PAYLOAD['round_count']} rounds),",
      f"conversation {CONVERSATION_PAYLOAD['round_id']}")


Fixtures loaded: talk_dna v1, vault v2 for Jamie, pulse v1 (2 rounds), conversation round_step13_demo


In [6]:
# ── 5. Results recorder ────────────────────────────
# Each check records (name, ok, detail) so the final summary cell
# can render unconditionally even when an earlier check fails.
RESULTS: list[dict] = []

def _record(name: str, ok: bool, detail: str) -> None:
    RESULTS.append({"check": name, "ok": ok, "detail": detail})
    print(f"{'PASS' if ok else 'FAIL'}  {name:42s} {detail}")

In [7]:
# ── 6. In-process Python API: round-trip every entity ────────
# Same payloads we'll later push through the HTTP API — verify the
# runtime contract before we add the FastAPI hop.

# 6a. TalkDNA
try:
    target = save_talk_dna(STORAGE_ROOT, TALK_DNA_PAYLOAD)
    loaded = load_talk_dna(STORAGE_ROOT, user_id="local")
    assert loaded == TALK_DNA_PAYLOAD, loaded
    assert target.exists() and target.name == "local.json"
    _record("runtime: TalkDNA save/load", True, f"-> {target.relative_to(STORAGE_ROOT)}")
except Exception as exc:  # noqa: BLE001
    _record("runtime: TalkDNA save/load", False, f"{type(exc).__name__}: {exc}")

# 6b. PersonVault
try:
    target = save_person_vault(STORAGE_ROOT, VAULT_PAYLOAD)
    loaded = load_person_vault(STORAGE_ROOT, VAULT_PAYLOAD["person_id"])
    assert loaded == VAULT_PAYLOAD
    listed = list_person_vaults(STORAGE_ROOT)
    assert len(listed) == 1 and listed[0]["person_id"] == VAULT_PAYLOAD["person_id"]
    _record("runtime: PersonVault save/load/list", True, f"-> {target.relative_to(STORAGE_ROOT)}")
except Exception as exc:  # noqa: BLE001
    _record("runtime: PersonVault save/load/list", False, f"{type(exc).__name__}: {exc}")

# 6c. Pulse
try:
    target = save_pulse(STORAGE_ROOT, PULSE_PAYLOAD)
    loaded = load_pulse(STORAGE_ROOT, PULSE_PAYLOAD["person_id"])
    assert loaded["round_count"] == 2
    assert loaded["health_trend"] == "improving"
    _record("runtime: Pulse save/load", True, f"v{loaded['version']} trend={loaded['health_trend']}")
except Exception as exc:  # noqa: BLE001
    _record("runtime: Pulse save/load", False, f"{type(exc).__name__}: {exc}")

# 6d. Conversation
try:
    target = save_conversation(STORAGE_ROOT, CONVERSATION_PAYLOAD)
    loaded = load_conversation(STORAGE_ROOT, CONVERSATION_PAYLOAD["round_id"])
    assert loaded == CONVERSATION_PAYLOAD
    listed = list_conversations(STORAGE_ROOT, person_id=VAULT_PAYLOAD["person_id"])
    assert len(listed) == 1 and listed[0]["round_id"] == CONVERSATION_PAYLOAD["round_id"]
    _record("runtime: Conversation save/load/list", True, f"-> {target.relative_to(STORAGE_ROOT)}")
except Exception as exc:  # noqa: BLE001
    _record("runtime: Conversation save/load/list", False, f"{type(exc).__name__}: {exc}")

# 6e. Missing record -> StorageNotFoundError
try:
    raised = False
    try:
        load_pulse(STORAGE_ROOT, "person_does_not_exist")
    except StorageNotFoundError:
        raised = True
    assert raised, "expected StorageNotFoundError"
    _record("runtime: missing record raises NotFound", True, "StorageNotFoundError raised")
except Exception as exc:  # noqa: BLE001
    _record("runtime: missing record raises NotFound", False, f"{type(exc).__name__}: {exc}")

# 6f. Schema validation catches missing required fields on write
try:
    raised = False
    bad = {k: v for k, v in TALK_DNA_PAYLOAD.items() if k != "patterns"}
    try:
        save_talk_dna(STORAGE_ROOT, bad)
    except StorageError as exc:
        raised = "patterns" in str(exc)
    assert raised, "expected StorageError mentioning the missing required field"
    _record("runtime: schema validation on write", True, "missing 'patterns' rejected")
except Exception as exc:  # noqa: BLE001
    _record("runtime: schema validation on write", False, f"{type(exc).__name__}: {exc}")

# 6g. Atomic-write invariant: no .tmp stragglers
try:
    stragglers = list(STORAGE_ROOT.rglob(".*.tmp"))
    assert stragglers == [], stragglers
    _record("runtime: atomic-write no .tmp stragglers", True, "0 leftover temp files")
except Exception as exc:  # noqa: BLE001
    _record("runtime: atomic-write no .tmp stragglers", False, f"{type(exc).__name__}: {exc}")


PASS  runtime: TalkDNA save/load                 -> talk_dna/local.json
PASS  runtime: PersonVault save/load/list        -> person_vault/person_cf528d57b789.json
PASS  runtime: Pulse save/load                   v1 trend=improving
PASS  runtime: Conversation save/load/list       -> conversations/round_step13_demo.json
PASS  runtime: missing record raises NotFound    StorageNotFoundError raised
PASS  runtime: schema validation on write        missing 'patterns' rejected
PASS  runtime: atomic-write no .tmp stragglers   0 leftover temp files


In [8]:
# ── 7. Build a TestClient with the storage root override ───────
# No model registry needed — the /storage/* routes are filesystem-
# only. We construct the TestClient WITHOUT a `with` block so the
# lifespan event (which loads Gemma 4) does NOT run; the storage
# root is injected via app.dependency_overrides.

# Use a fresh sub-directory so the HTTP-side checks don't see the
# Python-side fixtures from cell 6 — every assertion below starts
# from an empty layout.
HTTP_STORAGE_ROOT = STORAGE_ROOT / "http_root"
HTTP_STORAGE_ROOT.mkdir()
app.dependency_overrides[get_storage_root] = lambda: HTTP_STORAGE_ROOT

client = TestClient(app)
print(f"HTTP storage root            : {HTTP_STORAGE_ROOT}")
print("Storage routes exposed:")
for r in sorted(app.routes, key=lambda r: getattr(r, 'path', '')):
    path = getattr(r, 'path', '')
    if not path.startswith('/storage'):
        continue
    methods = ','.join(sorted(getattr(r, 'methods', set()) - {'HEAD'}))
    print(f"  {methods:8s} {path}")

HTTP storage root            : /tmp/step13_storage_yglqmt03/http_root
Storage routes exposed:
  GET      /storage/conversations
  GET      /storage/conversations/{round_id}
  PUT      /storage/conversations/{round_id}
  DELETE   /storage/conversations/{round_id}
  GET      /storage/pulse
  GET      /storage/pulse/{person_id}
  PUT      /storage/pulse/{person_id}
  GET      /storage/talk-dna
  PUT      /storage/talk-dna
  GET      /storage/vault
  GET      /storage/vault/{person_id}
  PUT      /storage/vault/{person_id}


In [9]:
# ── 8. HTTP round-trip every /storage/* endpoint ────────────
# Each block records (endpoint, ok, detail) into the same RESULTS
# list so the final cell renders a single combined table.

# 8a. /health surfaces the storage_root
try:
    r = client.get("/health")
    assert r.status_code == 200, r.text
    body = r.json()
    assert body["status"] == "ok"
    _record("http: GET /health", True, f"storage_root={body.get('storage_root')!r}")
except Exception as exc:  # noqa: BLE001
    _record("http: GET /health", False, f"{type(exc).__name__}: {exc}")

# 8b. PUT then GET TalkDNA
try:
    put = client.put("/storage/talk-dna", json=TALK_DNA_PAYLOAD)
    assert put.status_code == 200, put.text
    body = put.json()
    assert body["payload"]["user_id"] == "local"
    assert body["path"].endswith("local.json")
    got = client.get("/storage/talk-dna", params={"user_id": "local"})
    assert got.status_code == 200 and got.json() == TALK_DNA_PAYLOAD
    _record("http: PUT/GET /storage/talk-dna", True, f"v{got.json()['version']} stored")
except Exception as exc:  # noqa: BLE001
    _record("http: PUT/GET /storage/talk-dna", False, f"{type(exc).__name__}: {exc}")

# 8c. PUT then GET PersonVault, then LIST
try:
    put = client.put(f"/storage/vault/{VAULT_PAYLOAD['person_id']}", json=VAULT_PAYLOAD)
    assert put.status_code == 200, put.text
    got = client.get(f"/storage/vault/{VAULT_PAYLOAD['person_id']}")
    assert got.status_code == 200 and got.json()["name"] == "Jamie"
    listing = client.get("/storage/vault")
    assert listing.status_code == 200
    body = listing.json()
    assert body["count"] == 1 and body["items"][0]["person_id"] == VAULT_PAYLOAD["person_id"]
    _record("http: PUT/GET/LIST /storage/vault", True, f"1 vault listed for {got.json()['name']}")
except Exception as exc:  # noqa: BLE001
    _record("http: PUT/GET/LIST /storage/vault", False, f"{type(exc).__name__}: {exc}")

# 8d. PUT then GET Pulse, then LIST
try:
    put = client.put(f"/storage/pulse/{PULSE_PAYLOAD['person_id']}", json=PULSE_PAYLOAD)
    assert put.status_code == 200, put.text
    got = client.get(f"/storage/pulse/{PULSE_PAYLOAD['person_id']}")
    assert got.status_code == 200 and got.json()["round_count"] == 2
    listing = client.get("/storage/pulse")
    assert listing.json()["count"] == 1
    _record("http: PUT/GET/LIST /storage/pulse", True, (
        f"trend={got.json()['health_trend']}, score={got.json()['health_score']:.2f}"
    ))
except Exception as exc:  # noqa: BLE001
    _record("http: PUT/GET/LIST /storage/pulse", False, f"{type(exc).__name__}: {exc}")

# 8e. Full lifecycle for Conversations (PUT, GET, LIST, filter, DELETE)
try:
    put = client.put(
        f"/storage/conversations/{CONVERSATION_PAYLOAD['round_id']}",
        json=CONVERSATION_PAYLOAD,
    )
    assert put.status_code == 200, put.text
    got = client.get(f"/storage/conversations/{CONVERSATION_PAYLOAD['round_id']}")
    assert got.json()["round_id"] == CONVERSATION_PAYLOAD["round_id"]
    listing = client.get("/storage/conversations")
    assert listing.json()["count"] == 1
    filtered = client.get("/storage/conversations", params={"person_id": VAULT_PAYLOAD["person_id"]})
    assert filtered.json()["count"] == 1
    missed = client.get("/storage/conversations", params={"person_id": "person_nope"})
    assert missed.json()["count"] == 0
    delete = client.delete(f"/storage/conversations/{CONVERSATION_PAYLOAD['round_id']}")
    assert delete.status_code == 204
    after = client.get(f"/storage/conversations/{CONVERSATION_PAYLOAD['round_id']}")
    assert after.status_code == 404
    _record("http: conversations full lifecycle", True, "PUT/GET/LIST/filter/DELETE all green")
except Exception as exc:  # noqa: BLE001
    _record("http: conversations full lifecycle", False, f"{type(exc).__name__}: {exc}")

# 8f. Missing record -> 404
try:
    r = client.get("/storage/vault/person_does_not_exist")
    assert r.status_code == 404, r.text
    detail = r.json()["detail"]
    assert detail["component"] == "vault_storage"
    _record("http: missing record -> 404", True, f"component={detail['component']}")
except Exception as exc:  # noqa: BLE001
    _record("http: missing record -> 404", False, f"{type(exc).__name__}: {exc}")

# 8g. Bad payload -> 422
try:
    bad = {k: v for k, v in TALK_DNA_PAYLOAD.items() if k != "patterns"}
    r = client.put("/storage/talk-dna", json=bad)
    assert r.status_code == 422, r.text
    detail = r.json()["detail"]
    assert detail["component"] == "talk_dna_storage"
    assert "patterns" in detail["error"]
    _record("http: bad payload -> 422", True, f"component={detail['component']}")
except Exception as exc:  # noqa: BLE001
    _record("http: bad payload -> 422", False, f"{type(exc).__name__}: {exc}")


PASS  http: GET /health                          storage_root=None
PASS  http: PUT/GET /storage/talk-dna            v1 stored
PASS  http: PUT/GET/LIST /storage/vault          1 vault listed for Jamie
PASS  http: PUT/GET/LIST /storage/pulse          trend=improving, score=0.65
PASS  http: conversations full lifecycle         PUT/GET/LIST/filter/DELETE all green
PASS  http: missing record -> 404                component=vault_storage
PASS  http: bad payload -> 422                   component=talk_dna_storage


In [10]:
# ── 9. Inspect the on-disk layout ──────────────────────
# Quick visual confirmation that the directory shape on disk matches
# the contract documented in backend/core/_runtime/storage.py.

def _tree(root: Path, prefix: str = "") -> None:
    entries = sorted(root.iterdir()) if root.is_dir() else []
    for i, entry in enumerate(entries):
        connector = "└── " if i == len(entries) - 1 else "├── "
        print(f"{prefix}{connector}{entry.name}")
        if entry.is_dir():
            extension = "    " if i == len(entries) - 1 else "│   "
            _tree(entry, prefix + extension)

print("In-process (Python API) root:")
_tree(STORAGE_ROOT)
print()
print("HTTP (TestClient) root:")
_tree(HTTP_STORAGE_ROOT)

In-process (Python API) root:
├── conversations
│   └── round_step13_demo.json
├── http_root
│   ├── conversations
│   ├── person_vault
│   │   └── person_cf528d57b789.json
│   ├── pulse
│   │   └── person_cf528d57b789.json
│   └── talk_dna
│       └── local.json
├── person_vault
│   └── person_cf528d57b789.json
├── pulse
│   └── person_cf528d57b789.json
└── talk_dna
    └── local.json

HTTP (TestClient) root:
├── conversations
├── person_vault
│   └── person_cf528d57b789.json
├── pulse
│   └── person_cf528d57b789.json
└── talk_dna
    └── local.json


In [11]:
# ── 10. Final pass/fail summary (renders unconditionally) ───────
# Single combined table across the Python API checks (cell 6) and
# the HTTP API checks (cell 8). The cell never asserts — a single
# bad check should not hide the others.

if not RESULTS:
    print("No results recorded — the earlier cells did not run.")
else:
    name_w = max(len(r["check"]) for r in RESULTS)
    header = f"  {'check':{name_w}s}  status  detail"
    rule = "  " + "-" * (name_w + 8) + "-" * 60
    print(header)
    print(rule)
    for row in RESULTS:
        mark = "PASS" if row["ok"] else "FAIL"
        print(f"  {row['check']:{name_w}s}  {mark:6s}  {row['detail']}")
    print(rule)
    passed = sum(1 for r in RESULTS if r["ok"])
    total = len(RESULTS)
    print(f"  {passed}/{total} checks OK")

  check                                     status  detail
  ------------------------------------------------------------------------------------------------------------
  runtime: TalkDNA save/load                PASS    -> talk_dna/local.json
  runtime: PersonVault save/load/list       PASS    -> person_vault/person_cf528d57b789.json
  runtime: Pulse save/load                  PASS    v1 trend=improving
  runtime: Conversation save/load/list      PASS    -> conversations/round_step13_demo.json
  runtime: missing record raises NotFound   PASS    StorageNotFoundError raised
  runtime: schema validation on write       PASS    missing 'patterns' rejected
  runtime: atomic-write no .tmp stragglers  PASS    0 leftover temp files
  http: GET /health                         PASS    storage_root=None
  http: PUT/GET /storage/talk-dna           PASS    v1 stored
  http: PUT/GET/LIST /storage/vault         PASS    1 vault listed for Jamie
  http: PUT/GET/LIST /storage/pulse         PASS    tren